> 📓 **Lesson 1.8 — Part 4 of 4: Reading and Writing Data (Self-Study)**
>
> This notebook was split out of the original single `eda_basic.ipynb` so each part can be opened and run on its own. If you are starting here rather than at Part 1, run the **Setup** cell below first — it loads the same data used throughout Lesson 1.8.
>
> Other notebooks in this set: `Part_1_descriptive_statistics.ipynb`, `Part_2_data_quality.ipynb`, `Part_3_data_transformation.ipynb`

# Lesson 1.8: EDA Basic

Welcome to Exploratory Data Analysis. This notebook takes one raw, messy business file and walks the
full path: understanding its structure, cleaning it, transforming it, and moving it in and out of
files.

**Structure — the four learning outcomes, in order:**
* **Part 1: Descriptive Statistics** — *summarise* a dataset: shape, data types, distributions.
* **Part 2: Data Quality** — *handle* the messy reality: missing values, duplicates, impossible values.
* **Part 3: Data Transformation** — *transform* for analysis: mapping, labels, strings, categories, dates.
* **Part 4: Reading & Writing Data** — *read and write* CSV, JSON, Excel, databases.

**For Learners:** read the `# 👉` comment above each line before you run the cell. The comment says
what the line does in plain English; the output shows you it happened.


> **🧭 Today's flow — 150 minutes.** One messy file, four learning outcomes, in order:
>
> | | Section | Learning outcome | Time |
> |---|---|---|---|
> | — | Setup + why this matters | | 5 min |
> | **Part 1** | Descriptive Statistics | **Summarise** a dataset: shape, dtypes, distributions | 33 min |
> | ☕ | *Break* | | 10 min |
> | **Part 2** | Data Quality | **Handle** missing values, duplicates, impossible values | 42 min |
> | ☕ | *Break* | | 10 min |
> | **Part 3** | Data Transformation | **Transform**: mapping, labels, strings, categories, dates, grouping | 35 min |
> | **Part 4** | Reading & Writing Data | **Read and write** CSV, JSON, Excel, databases | 15 min |
>
> **The spine:** we work on one file, `data/cafe_june_raw.csv`, from start to finish. Each section
> improves the same `clean` table, and Part 4 saves it. Small hand-built tables appear alongside it
> as *drills* — they isolate one method so you can see exactly what it does.
>
> Each of Parts 1–3 ends with a **🛠️ Group Exercise**. Deep dives live in `reference.md`;
> the Appendix at the end is self-study.


### The business problem

> **The Daily Grind** is a four-outlet café chain in Singapore. Revenue has been flat for two
> quarters, and the owner has to decide whether to renew the Marina Bay lease. She asks her
> assistant to send you the sales data. What arrives is a **raw till export**: one row per outlet,
> per day, per part of the day, straight out of the point-of-sale system, untouched.
>
> Nobody can answer the owner's question from this file yet. Today's job is to make it
> answerable — and to be able to say *why* every number in it can be trusted.

This is the first of three lessons on the same problem:

| Lesson | The question | What you do |
|---|---|---|
| **1.8 — today** | **Can I trust this data?** | clean one month of the raw export |
| 1.9 | What is the pattern? | 18 months, cleaned: time, joins, grouping |
| 1.10 | How do I make them act? | one chart, one slide, one decision |


### Setup

Import the libraries, then load the file we will use all session.


In [ ]:
# 👉 Load the two toolkits we need. `pd` and `np` are just short nicknames so we can
#    type `pd.something` instead of `pandas.something`. Run this cell first, every session.
import pandas as pd
import numpy as np


In [ ]:
# 👉 Load the dataset we will use all session: June 2025's till export from four cafés.
#    `read_csv` reads a comma-separated text file into a DataFrame -- a table with named columns.
#    pandas already treats an empty field, "NA" and "n/a" as missing.
raw = pd.read_csv("../data/cafe_june_raw.csv")

raw


### 🎬 Why this matters — before you trust a single number

Run the next three cells. The chain has **four** cafés and its busiest shift takes about \$1,000.


In [ ]:
# 👉 `.value_counts()` counts how many rows have each value. How many cafés do you count?
raw["outlet"].value_counts()


In [ ]:
# 👉 The same question of the daypart column. There are three parts to a trading day.
raw["daypart"].value_counts()


In [ ]:
# 👉 Sort the takings column and look at the two ends. `.dropna()` skips the blank cells,
#    because a sort cannot compare text with a blank -- which is itself a clue.
#    `.iloc[[0, -1]]` takes the first and last rows of the sorted result.
raw["revenue_raw"].dropna().sort_values().iloc[[0, -1]]


**Three problems, in three lines of output.**

1. **Twelve spellings for four cafés.** `Raffles Place`, `raffles place`, `RAFFLES PLACE`,
   `Raffles Pl.`… Group by outlet today and you get twelve cafés, four of which are the same shop.
2. **Nine labels for three dayparts** — `Morning`, `morning`, `AM`, and so on.
3. **The revenue column is not a number.** Sorted, the "smallest" value is `" 1,006.71 "` and the
   "largest" is `"S$94.41"`, because pandas is comparing them as **text**: a space sorts before a
   digit, and the letter `S` sorts after every digit. Sorted as text, \$98,000 loses to \$99.

Any average, chart or model built on this file is wrong before you start. Worse, none of it would
*look* wrong: it would produce numbers, with decimal places, and nobody in the meeting would know.

Part 1 is the routine that finds problems like these in about two minutes.


---

## Part 4: Reading and Writing Data (Self-Study)

**Learning outcome 4:** *Read and write data across multiple file formats (CSV, JSON, Excel,
databases).*

**Goal:** get the cleaned table out of this notebook and into a file somebody else can use.
Nothing in Parts 1–3 needed this, which is why it comes last.

⏱️ ~15 min


### 4.1: Reading Data

Pandas is flexible about input formats. Start with a plain comma-separated file.


In [ ]:
# 👉 The leading `!` runs a terminal command rather than Python. `cat` prints a file as-is,
#    which is the fastest way to see what you are about to load.
!cat ../data/ex1.csv


In [ ]:
# 👉 `read_csv` turns that text file into a DataFrame. The path `../data/` means
#    'go up one folder from notebooks/, then into data/'.
pd.read_csv("../data/ex1.csv")


**Scenario:** what if the file has no header row? Without telling pandas, it will use your first
row of real data as the column names.


In [ ]:
# 👉 Same peek, at a file whose first line is real data rather than column names.
!cat ../data/ex2.csv


In [ ]:
# 👉 `header=None` says 'there are no column names', so pandas numbers them 0, 1, 2...
pd.read_csv("../data/ex2.csv", header=None)


In [ ]:
# 👉 Better: supply your own column names with `names=`.
pd.read_csv(
    "../data/ex2.csv",
    names=["date", "outlet_id", "daypart", "tickets", "items", "revenue_sgd"],
)


**Indexing:** you can promote a column to be the row labels.


In [ ]:
# 👉 `index_col=` promotes one column to be the row labels instead of an ordinary column.
names = ["date", "outlet_id", "daypart", "tickets", "items", "revenue_sgd"]

pd.read_csv("../data/ex2.csv", names=names, index_col="outlet_id")


**Junk before the header:** exports often start with a couple of comment lines.


In [ ]:
# 👉 Two lines of preamble before the real header.
!cat ../data/ex4.csv


In [ ]:
# 👉 `skiprows=` drops lines by position. `comment="#"` is the more robust version:
#    it ignores any line starting with that character, wherever it appears.
pd.read_csv("../data/ex4.csv", comment="#")


**Handling missing values at load time:** pandas already recognises empty fields, `NA`, `NULL` and
`n/a`. It cannot guess *your* system's conventions.


In [ ]:
# 👉 This file writes missing values in several different ways.
!cat ../data/ex5.csv


In [ ]:
# 👉 Out of the box, pandas recognises the empty field, 'NA', 'NULL' and 'n/a' -- but -999
#    comes through as a number, because there is no way for pandas to know it is a code.
result = pd.read_csv("../data/ex5.csv")

result


In [ ]:
# 👉 `na_values=` adds your own conventions to the list. Doing it here, at load time, is
#    tidier than fixing it later -- and it means the sentinel never poisons a statistic.
pd.read_csv("../data/ex5.csv", na_values=["-999"])


In [ ]:
# 👉 Sometimes 'missing' is spelled differently per column. A dictionary says which
#    text counts as missing in which column.
pd.read_csv("../data/ex5.csv", na_values={"tickets": ["-999"], "revenue_sgd": ["n/a"]})


**Reading Excel:** a workbook can hold several sheets, so you usually look before you load.


In [ ]:
# 👉 Open the workbook and see what is inside it.
workbook = pd.ExcelFile("../data/cafe_june_workbook.xlsx")

workbook.sheet_names


In [ ]:
# 👉 `.parse()` reads one named sheet into a DataFrame.
workbook.parse(sheet_name="Outlets")


In [ ]:
# 👉 The one-line shortcut when you already know the sheet you want.
pd.read_excel("../data/cafe_june_workbook.xlsx", sheet_name="June").head()


In [ ]:
# 👉 Now compare. This workbook (and the database in 4.3) holds the *reference* clean June --
#    the version with the eight damaged shifts restored from the original till logs.
reference = pd.read_excel("../data/cafe_june_workbook.xlsx", sheet_name="June")

print(f"your cleaned June:      ${clean['revenue_sgd'].sum():>11,.2f}")
print(f"reference clean June:   ${reference['revenue_sgd'].sum():>11,.2f}")
print(f"difference:             ${reference['revenue_sgd'].sum() - clean['revenue_sgd'].sum():>11,.2f}")


> **Your number is different, and that is the right answer.** Eight shifts in the raw export had lost
> their true figure — four sentinels, one mis-key, three blanks — and you filled them with the median,
> because the median is the best defensible guess available *from that file*. The reference version was
> reconstructed from the original till logs, which you were never given.
>
> \$916 on \$175,669 is **0.5%**, and you can name every dollar of it. That is what a clean dataset
> looks like in practice: not identical to the truth, but different from it in ways you can explain.
>
> This reference file is the June slice of what Lesson 1.9 opens next.


### 4.2: Writing Data (Exporting)

Saving is the mirror of reading.


In [ ]:
# 👉 `to_csv` writes the DataFrame back out to a file.
result.to_csv("../data/out.csv")

# 👉 Peek at what was written. Notice the extra unnamed first column -- that is the index.
!cat ../data/out.csv


**Tip:** you usually want `index=False`, so the row numbers are not saved as a mystery column.


In [ ]:
# 👉 `index=False` leaves the row labels out, which is what you want when the index is
#    just 0, 1, 2... and carries no meaning.
result.to_csv("../data/out.csv", index=False)

!cat ../data/out.csv


**JSON export:** the format APIs and web services speak.


In [ ]:
# 👉 `orient="records"` writes a list of one object per row, which is what most web APIs expect.
result.to_json("../data/out.json", orient="records")

!cat ../data/out.json


In [ ]:
# 👉 And straight back in.
pd.read_json("../data/out.json", orient="records")


**Save our session's work.** Everything from Parts 2 and 3 lives in `clean`. Write out the columns
a colleague would actually want.


In [ ]:
# 👉 Keep the useful columns, in a sensible order, with the tidy names.
final = clean[[
    "date", "outlet_name", "daypart", "tickets", "items", "revenue_sgd", "staff", "manager",
]].sort_values(["date", "outlet_name", "daypart"])

final.to_csv("../data/cafe_june_clean.csv", index=False)

final.head()


In [ ]:
# 👉 Beat 4 of the 1.8 habit -- verify. Read back what you just wrote and check it matches.
check = pd.read_csv("../data/cafe_june_clean.csv")

print("rows written:", len(check))
print(f"June revenue: ${check['revenue_sgd'].sum():,.2f}")
print("date column came back as:", check["date"].dtype)


> **Note that last line.** The date went out as a real date and came back as **text**. CSV has no
> way to store types, so every reader has to re-parse them (`parse_dates=["date"]`). Databases and
> pickle files remember types; CSV and JSON forget them. That is worth knowing before you build a
> pipeline out of CSVs.


**Excel export:**


In [ ]:
# 👉 Writing Excel needs a 'writer' object when you want control over sheets.
#    Using `with` closes the file for you, which is what actually saves it.
with pd.ExcelWriter("../data/cafe_june_clean.xlsx") as writer:
    final.to_excel(writer, sheet_name="June", index=False)
    june_summary.to_excel(writer, sheet_name="Summary")

pd.ExcelFile("../data/cafe_june_clean.xlsx").sheet_names


### 4.3: Databases

Most real data lives in a database rather than a file. `sqlalchemy` is the library pandas uses to
talk to them, and the pandas side barely changes.


In [ ]:
# 👉 SQLAlchemy speaks to many database engines through one interface.
import sqlalchemy as sqla

# 👉 A connection string says which engine and which file. `sqlite:///` is a local file --
#    no server to install, which is why it is the right thing for a lesson.
engine = sqla.create_engine("sqlite:///../data/cafe.db")

# 👉 What tables are in there? `inspect` is the version-safe way to ask.
sqla.inspect(engine).get_table_names()


In [ ]:
# 👉 Give a table name and pandas reads the whole table into a DataFrame.
pd.read_sql_table("outlets", engine)


In [ ]:
# 👉 Or hand it real SQL and only the query result comes back. Same DataFrame either way.
#    Everything you learned about SQL in Lessons 1.3-1.5 works here.
query = (
    'SELECT outlet_id, SUM(revenue_sgd) AS revenue, COUNT(*) AS shifts '
    'FROM daily_sales_june '
    'GROUP BY outlet_id '
    'ORDER BY revenue DESC'
)

pd.read_sql(query, engine)


In [ ]:
# 👉 Writing back: `to_sql` creates (or replaces) a table from a DataFrame.
#    This is how a cleaned table gets handed to the rest of the business.
final.to_sql("june_clean", engine, index=False, if_exists="replace")

sqla.inspect(engine).get_table_names()


> **Task:** write a *filtered* table to the database.
> 1. Filter `final` to Marina Bay only.
> 2. Write it to a new table called `marina_june`.
> 3. Read it back with `pd.read_sql` and check the row count is 90 (30 days × 3 dayparts).


> **Final challenge:**
> 1. Read `daily_sales_june` back out of the database.
> 2. Compute total revenue per daypart with SQL (`GROUP BY daypart`) *and* with pandas
>    (`groupby("daypart")`).
> 3. Confirm the two agree. Two tools, one answer — that is the check worth building the habit on.


## 🎯 Wrap-Up

1. **Always run the five-move first look** on a new file — head, shape, info, dtypes, describe.
   It takes two minutes and it found all five of today's problems.
2. **Wrong types hide everything.** The impossible \$98,000 and the -999 sentinels were invisible
   while revenue was text. Move 4 (`.dtypes`) is not paperwork.
3. **Cleaning decisions depend on what the value means**, not on what is convenient: fill, drop,
   cap or mark missing are four different answers to four different situations.
4. **Order of operations matters.** Remove the sentinels *before* you compute the median you are
   going to impute with.
5. **Standardise your keys before you trust a check.** The duplicate check in 2.2 was meaningless
   until the twelve outlet spellings became four.
6. **A clean table is not the goal; a trustworthy answer is.** Section 3.5 — four rows, one per
   café — is what all the cleaning was for.

**Next Steps:**
- Complete the [Assignment](./assignment.md) — audit a second month of the same export.
- Next lesson: **1.9 EDA Advanced** opens `daily_sales.csv` — the same export for **all 18 months**,
  cleaned exactly the way you just cleaned June, and asks what the pattern is.

> **The handoff, in one number.** Your cleaned June total is **\$174,753**. In Lesson 1.9's clean
> 18-month file, June 2025 is **\$175,669** — a difference of **\$916**, or 0.5%, and you can say
> exactly where it comes from: **eight shifts** (four sentinels, one mis-keyed figure and three blanks)
> where you put the median in place of a number this export had destroyed.
>
> That is the difference between clean data and lucky data: not that your figure matches, but that
> you can account for why it does not.


---

## 📎 Appendix — Self-Study

These topics fall outside today's four learning outcomes, but come up constantly in practice.
Deep dives on categorical internals, awkward CSV parsing and pickle files live in `reference.md`.


### Permutation and Random Sampling

Reordering rows, or taking a random subset — the basis of train/test splits later in the course.


In [ ]:
# 👉 A 5x7 table of the numbers 0-34 to make row shuffling easy to see.
df = pd.DataFrame(np.arange(5 * 7).reshape((5, 7)))

df


In [ ]:
# 👉 `permutation(5)` returns the numbers 0-4 in random order -- a shuffled seating plan.
sampler = np.random.default_rng(seed=12345).permutation(5)

sampler


In [ ]:
# 👉 `.iloc[...]` selects rows *by position*, so the table comes back in the shuffled order.
df.iloc[sampler]


In [ ]:
# 👉 `.take()` does the same job and reads a little more clearly.
df.take(sampler)


Permuting columns:


In [ ]:
# 👉 `df.shape[1]` is the number of columns, so this shuffles the column positions.
column_sampler = np.random.default_rng(seed=12345).permutation(df.shape[1])

column_sampler


In [ ]:
# 👉 `axis=1` applies the shuffle to columns instead of rows.
df.take(column_sampler, axis=1)


**Random sample:** getting a random subset without the manual shuffle.


In [ ]:
# 👉 `.sample()` skips the manual shuffle: just ask for 3 random rows.
df.sample(n=3)


In [ ]:
# 👉 `replace=True` puts each pick back in the hat, so 10 draws from 5 rows is possible
#    and rows can repeat. This is 'sampling with replacement'.
df.sample(n=10, replace=True)


> **Task:** sample `df` using the parameter `frac` (a fraction of the rows) instead of `n` (a count).


---

## 🏁 Lesson 1.8 Complete!

You've covered all four sections:
- **Part 1:** Descriptive Statistics — summarising a dataset
- **Part 2:** Data Quality — missing values, duplicates, impossible values
- **Part 3:** Data Transformation — mapping, labels, strings, categories, dates
- **Part 4:** Reading & Writing Data — CSV, JSON, Excel, databases

**Next step:** work through `assignment.md` — audit a second month of the same export.
Then open **Lesson 1.9 — EDA Advanced**, which loads `daily_sales.csv`, the same export cleaned this way for all 18 months, and asks what the pattern is.